# Phase 4, Stage 3a: Kolmogorov-Smirnov Tests (primary test)

## What the KS test does

The two-sample Kolmogorov-Smirnov (KS) test compares two distributions **without assuming any particular shape** (unlike a t-test, which focuses on means and assumes roughly normal data). It looks at the biggest vertical gap between the two samples' cumulative distributions — this gap is the **D statistic**.

- **D** ranges from 0 (distributions identical) to 1 (distributions completely non-overlapping). This is your **effect size** — how *different* the shapes are.
- The **p-value** tells you whether that gap is unlikely to have occurred by chance alone, given the sample sizes.

This is the **primary test** for this project because it's sensitive to *any* kind of distributional difference — shifts in location, spread, skew, or tail behaviour — which directly matches the SRQ's focus on the *shape* of the CPL distribution.

## What we're comparing

For each of the 5 rating bands, we compare `capped_cpl` between **adjacent** time pressure bins:
- Bin 1 (Minimal) vs Bin 2 (Low)
- Bin 2 (Low) vs Bin 3 (Moderate)
- Bin 3 (Moderate) vs Bin 4 (High)

That's 3 comparisons x 5 bands = **15 tests total**.

## Multiple comparisons correction

Running 15 tests increases the chance of a false positive (a "significant" result that's actually just noise) just by chance. The **Bonferroni correction** divides the usual significance threshold (α = 0.05) by the number of tests:

**Corrected α = 0.05 / 15 = 0.0033**

A result is only treated as statistically significant if p < 0.0033.

## A caution on sample size

Our cells range from ~16,500 to ~72,000 moves. With samples this large, the KS test will detect *extremely* small differences as "significant" — so **p < 0.0033 will likely be true for almost every comparison**. The D statistic (the actual size of the distributional gap) is what tells us whether a difference is *meaningful*, not just detectable. We'll report both, but focus interpretation on D.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

ALPHA_BONFERRONI = 0.05 / 15  # 0.0033, for the 15 KS comparisons

RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}

TIME_PRESSURE_LABELS = {
    1: 'Minimal (>75%)',
    2: 'Low (50-75%)',
    3: 'Moderate (25-50%)',
    4: 'High (<25%)',
}

df = pd.read_csv('../../data/processed/analysed_moves.csv')
print(f'Loaded {len(df):,} rows')
print(f'Bonferroni-corrected alpha for KS tests: {ALPHA_BONFERRONI:.4f}')

## Running the 15 pairwise tests

For each rating band, we pull out the `capped_cpl` values for each adjacent pair of time pressure bins and run `scipy.stats.ks_2samp`.

In [ ]:
adjacent_transitions = [(1, 2), (2, 3), (3, 4)]

results = []
for rating_band in sorted(RATING_BAND_LABELS):
    for bin_a, bin_b in adjacent_transitions:
        sample_a = df.loc[(df['rating_band'] == rating_band) & (df['time_pressure_bin'] == bin_a), 'capped_cpl']
        sample_b = df.loc[(df['rating_band'] == rating_band) & (df['time_pressure_bin'] == bin_b), 'capped_cpl']

        ks_result = stats.ks_2samp(sample_a, sample_b)

        results.append({
            'rating_band': rating_band,
            'rating_band_label': RATING_BAND_LABELS[rating_band],
            'transition': f'Bin {bin_a} -> Bin {bin_b}',
            'bin_a_label': TIME_PRESSURE_LABELS[bin_a],
            'bin_b_label': TIME_PRESSURE_LABELS[bin_b],
            'n_a': len(sample_a),
            'n_b': len(sample_b),
            'D': ks_result.statistic,
            'p_value': ks_result.pvalue,
            'significant_bonferroni': ks_result.pvalue < ALPHA_BONFERRONI,
        })

ks_results = pd.DataFrame(results)
ks_results

## Save results and look at the D statistics by band/transition

We'll save the full table to `../results/` for the write-up, then pivot it so we can directly compare D values across rating bands for each transition — this is the input for the Stage 4.7 between-band comparison.

In [ ]:
ks_results.to_csv('../results/ks_test_results.csv', index=False)
print('Saved to ../results/ks_test_results.csv')

# Pivot: rows = rating band, columns = transition, values = D statistic
d_pivot = ks_results.pivot(index='rating_band_label', columns='transition', values='D')
d_pivot = d_pivot.reindex(index=[RATING_BAND_LABELS[b] for b in sorted(RATING_BAND_LABELS)],
                           columns=['Bin 1 -> Bin 2', 'Bin 2 -> Bin 3', 'Bin 3 -> Bin 4'])
d_pivot